# SageMaker Processing Job — BYOC con scikit-learn

Preprocesamiento del dataset **1C Company — Predict Future Sales (Kaggle)** usando un container
propio (BYOC) con scikit-learn.

### Flujo de datos

```
S3 (sales_train.csv, items.csv)  →  /opt/ml/processing/input/  →  preprocess.py  →  /opt/ml/processing/output/  →  S3 (datos procesados)
```

El script no lee ni escribe S3 directamente — solo trabaja con rutas locales dentro del container. 

### Contenidos
1. [Setup](#setup)
2. [Carga del dataset a S3](#dataset)
3. [Construcción del container](#container)
4. [Script de preprocessing](#script)
5. [Ejecutar el processing job](#run)
6. [Inspeccionar el output](#inspect)

## 1. Setup {#setup}

Sesión de SageMaker, IAM role, bucket y prefijos S3.

In [1]:
from time import gmtime, strftime
import boto3
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")

prefix = "sagemaker/1c-processing"
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

input_prefix  = prefix + "/input/raw"
output_prefix = prefix + "/input/preprocessed"

print(f"Role    : {role}")
print(f"Bucket  : {bucket}")
print(f"Region  : {region}")
print(f"Account : {account_id}")
print(f"Prefix  : {prefix}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Role    : arn:aws:iam::561137843164:role/SageMakerStudioExecutionRole2026
Bucket  : sagemaker-us-east-1-561137843164
Region  : us-east-1
Account : 561137843164
Prefix  : sagemaker/1c-processing


## 2. Carga del dataset a S3 {#dataset}

Sube `sales_train.csv` e `items.csv` al bucket de SageMaker como datos crudos.

**Aclaración**

Mi repo no tenía los datos en data/raw porque eran muy pesado, al clonar el repo solamente se mantuvo la estructura de las carpetas. 
Para subir los datos, los cargue manualmente a data/raw y es necesario verificar en qué carpeta estamos parados. 

In [10]:
import os
os.getcwd()

'/home/sagemaker-user/ModeloML-predsales'

En el caso que se esté en "notebook", correr la siguiente línea que te sube al directorio. 
Lo que estamos buscando es estar en:

- /home/sagemaker-user/ModeloML-predsales

In [ ]:
os.chdir("..")
print(os.getcwd())

In [8]:
for filename in ["sales_train.csv", "items.csv"]:
    uri = sagemaker_session.upload_data(
        path=f"data/raw/{filename}",
        bucket=bucket,
        key_prefix=input_prefix
    )
    print(f"Datos cargados: {uri}")

Datos cargados: s3://sagemaker-us-east-1-561137843164/sagemaker/1c-processing/input/raw/sales_train.csv
Datos cargados: s3://sagemaker-us-east-1-561137843164/sagemaker/1c-processing/input/raw/items.csv


## 3. Construcción del container {#container}

El container BYOC es una imagen `python:3.12-slim` con scikit-learn, pandas y numpy.
No requiere bootstrapping ni configuración de clusters — SageMaker inyecta y ejecuta el script directamente con `python3`.

In [11]:
%cd src/preprocessing
!docker build --network sagemaker -t 1c-preprocessing .
%cd ../..

/home/sagemaker-user/ModeloML-predsales/src/preprocessing
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  60.93kB
Step 1/10 : FROM python:3.12-slim
3.12-slim: Pulling from library/python

56c42440: Pulling fs layer 
d65d3b24: Pulling fs layer 
cc6599a6: Pulling fs layer 
Digest: sha256:ccc7089399c8bb65dd1fb3ed6d55efa538a3f5e7fca3f5988ac3b5b87e593bf0
Status: Downloaded newer image for python:3.12-slim
 ---> 6f90d4a79e7a
Step 2/10 : WORKDIR /app
 ---> Running in 6a41642f6e70
 ---> Removed intermediate container 6a41642f6e70
 ---> be9529c6cd68
Step 3/10 : COPY requirements.txt .
 ---> acea8d84cedf
Step 4/10 : RUN pip install --no-cache-dir -r requirements.txt
 ---> Running in 070fdf7ab7d8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 286.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

### Push a Amazon ECR

In [12]:
ecr_repository = "1c-preprocessing"
tag = ":latest"
uri_suffix = "amazonaws.com"

repository_uri = "{}.dkr.ecr.{}.{}/{}".format(
    account_id, region, uri_suffix, ecr_repository + tag
)

print(f"Image URI: {repository_uri}")

Image URI: 561137843164.dkr.ecr.us-east-1.amazonaws.com/1c-preprocessing:latest


In [13]:
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
!aws ecr create-repository --repository-name {ecr_repository}
!docker tag {ecr_repository + tag} {repository_uri}
!docker push {repository_uri}

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded
{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-east-1:561137843164:repository/1c-preprocessing",
        "registryId": "561137843164",
        "repositoryName": "1c-preprocessing",
        "repositoryUri": "561137843164.dkr.ecr.us-east-1.amazonaws.com/1c-preprocessing",
        "createdAt": "2026-03-13T21:10:46.033000+00:00",
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": false
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}
The push refers to repository [561137843164.dkr.ecr.us-east-1.amazonaws.com/1c-preprocessing]

4f0d1c2f: Preparing 
b9e8dda8: Preparing 
6532a578: Preparing 
1f172a68: Preparing 
f940005c: P

## 4. Script de preprocessing {#script}

La celda siguiente escribe `preprocess.py` en el directorio actual y los sube a S3.

**¿Qué hace el script?**

- **Carga:** lee `sales_train.csv` e `items.csv` desde `/opt/ml/processing/input/`.
- **Limpieza:** elimina registros con precio no positivo o unidades negativas.
- **Agregación mensual:** agrupa ventas diarias a nivel `date_block_num / shop_id / item_id` y clipea a 20.
- **Enriquecimiento:** agrega `item_category_id` desde `items.csv`.
- **Grid completo:** construye todas las combinaciones activas de shop/item/mes, rellenando con 0 donde no hubo ventas.
- **Lag features:** genera lags 1, 3, 6 y 12 meses del target.
- **Casos completos:** elimina filas con NaN en los lags (primeros meses sin historia).
- **Split train/test:** divide en 70% train / 30% test.
- **Output:** guarda 4 archivos CSV en `/opt/ml/processing/output/` que SageMaker copia a S3.

In [14]:
%%writefile preprocess.py
from __future__ import print_function, unicode_literals
import argparse
import os

import pandas as pd
from sklearn.model_selection import train_test_split

# ── Constantes ────────────────────────────────────────────────────────────────
COLUMN_UNITS       = "item_cnt_day"
COLUMN_UNITS_MONTH = "item_cnt_month"
COLUMN_PRICE       = "item_price"
CLIP               = 20
LAGS               = [1, 3, 6, 12]


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    args, _ = parser.parse_known_args()

    # ── Rutas SageMaker ───────────────────────────────────────────────────────
    input_dir  = "/opt/ml/processing/input"
    train_dir  = "/opt/ml/processing/output/train"
    test_dir   = "/opt/ml/processing/output/test"

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir,  exist_ok=True)

    # ── 1. Carga ──────────────────────────────────────────────────────────────
    print("Loading sales data...")
    sales = pd.read_csv(os.path.join(input_dir, "sales_train.csv"))
    print("Sales loaded: {} records".format(len(sales)))

    print("Loading items data...")
    items = pd.read_csv(os.path.join(input_dir, "items.csv"))
    print("Items loaded: {} records".format(len(items)))

    # ── 2. Limpieza ───────────────────────────────────────────────────────────
    print("Cleaning sales data...")
    sales = sales[(sales[COLUMN_PRICE] > 0) & (sales[COLUMN_UNITS] >= 0)].copy()
    print("After cleaning: {} records".format(len(sales)))

    # ── 3. Agregación mensual ─────────────────────────────────────────────────
    print("Aggregating to monthly level...")
    monthly = (
        sales.groupby(["date_block_num", "shop_id", "item_id"], as_index=False)
        .agg({COLUMN_UNITS: "sum"})
        .rename(columns={COLUMN_UNITS: COLUMN_UNITS_MONTH})
    )
    monthly[COLUMN_UNITS_MONTH] = monthly[COLUMN_UNITS_MONTH].clip(0, CLIP)
    print("Monthly aggregation: {} records".format(len(monthly)))

    # ── 4. Agrega item_category_id ────────────────────────────────────────────
    print("Adding item category information...")
    monthly = monthly.merge(
        items[["item_id", "item_category_id"]], on="item_id", how="left"
    )

    # ── 5. Grid completo ──────────────────────────────────────────────────────
    print("Building complete shop-item-month grid...")
    monthly_sorted = monthly.sort_values(["shop_id", "item_id", "date_block_num"])
    total_months = sorted(monthly_sorted["date_block_num"].unique())
    comb_active  = monthly_sorted[["shop_id", "item_id"]].drop_duplicates()

    grid = pd.concat(
        [comb_active.assign(date_block_num=m) for m in total_months],
        ignore_index=True
    )[["date_block_num", "shop_id", "item_id"]]

    grid = grid.merge(
        monthly_sorted[["date_block_num", "shop_id", "item_id",
                         COLUMN_UNITS_MONTH, "item_category_id"]],
        on=["date_block_num", "shop_id", "item_id"],
        how="left",
    )
    grid[COLUMN_UNITS_MONTH]   = grid[COLUMN_UNITS_MONTH].fillna(0)
    grid["item_category_id"]   = grid["item_category_id"].fillna(
        grid["item_category_id"].mode()[0]
    )
    grid = grid.sort_values(["shop_id", "item_id", "date_block_num"])
    print("Grid built: {} records".format(len(grid)))

    # ── 6. Lag features ───────────────────────────────────────────────────────
    print("Generating lag features: {}...".format(LAGS))
    for lag in LAGS:
        grid["lag_{}".format(lag)] = (
            grid.groupby(["shop_id", "item_id"])[COLUMN_UNITS_MONTH].shift(lag)
        )

    # ── 7. Filtra casos completos ─────────────────────────────────────────────
    lag_cols = ["lag_{}".format(l) for l in LAGS]
    grid = grid.dropna(subset=lag_cols)
    print("After filtering incomplete cases: {} records".format(len(grid)))

    # ── 8. Split temporal train / test ───────────────────────────────────────
    feature_cols = ["date_block_num", "shop_id", "item_id", "item_category_id"] + lag_cols
    max_block = grid["date_block_num"].max()  # mes 33
    print("Max date_block_num (validation month): {}".format(max_block))

    X_train = grid[grid["date_block_num"] < max_block][feature_cols]
    y_train = grid[grid["date_block_num"] < max_block][COLUMN_UNITS_MONTH]

    X_test  = grid[grid["date_block_num"] == max_block][feature_cols]
    y_test  = grid[grid["date_block_num"] == max_block][COLUMN_UNITS_MONTH]

    print("Train data shape: {}".format(X_train.shape))
    print("Test data shape : {}".format(X_test.shape))

    # ── 9. Guarda los 4 archivos de output ────────────────────────────────────
    train_features_path = os.path.join(train_dir, "train_features.csv")
    train_labels_path   = os.path.join(train_dir, "train_labels.csv")
    test_features_path  = os.path.join(test_dir,  "test_features.csv")
    test_labels_path    = os.path.join(test_dir,  "test_labels.csv")

    print("Saving train features to {}".format(train_features_path))
    X_train.to_csv(train_features_path, header=False, index=False)

    print("Saving test features to {}".format(test_features_path))
    X_test.to_csv(test_features_path, header=False, index=False)

    print("Saving train labels to {}".format(train_labels_path))
    y_train.to_csv(train_labels_path, header=False, index=False)

    print("Saving test labels to {}".format(test_labels_path))
    y_test.to_csv(test_labels_path, header=False, index=False)

    print("Preprocessing complete.")

Writing preprocess.py


## 5. Ejecutar el Processing Job {#run}

`ScriptProcessor` ejecuta el script en el container BYOC con una sola instancia.


In [19]:
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor

processor = ScriptProcessor(
    base_job_name="1c-preprocessing",
    image_uri=repository_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    max_runtime_in_seconds=1200,
)

processor.run(
    code="preprocess.py",
    inputs=[
        ProcessingInput(
            source="s3://{}/{}".format(bucket, input_prefix),
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
            destination="s3://{}/{}/train".format(bucket, output_prefix),
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
            destination="s3://{}/{}/test".format(bucket, output_prefix),
        ),
    ],
    logs=True,
)

INFO:sagemaker:Creating processing-job with name 1c-preprocessing-2026-03-13-21-44-23-882


.............Loading sales data...
Sales loaded: 2935849 records
Loading items data...
Items loaded: 22170 records
Cleaning sales data...
After cleaning: 2928492 records
Aggregating to monthly level...
Monthly aggregation: 1608226 records
Adding item category information...
Building complete shop-item-month grid...
Grid built: 14419332 records
Generating lag features: [1, 3, 6, 12]...
After filtering incomplete cases: 9330156 records
Max date_block_num (validation month): 33
Train data shape: (8906058, 8)
Test data shape : (424098, 8)
Saving train features to /opt/ml/processing/output/train/train_features.csv
Saving test features to /opt/ml/processing/output/test/test_features.csv
Saving train labels to /opt/ml/processing/output/train/train_labels.csv
Saving test labels to /opt/ml/processing/output/test/test_labels.csv
Preprocessing complete.



## 6. Inspeccionar el output {#inspect}

Vamos a revisar las primeras filas de los datos preparados para verificar que el preprocessing fue exitoso. Para eso haremos tres pruebas: 

- Las primeras 5 filas del archivo *train_features.csv*
- El tamaño de la base *train_features.csv* (esperamos 8.9M filas y 8 columnas)
- El tamaño de la base *test_features.csv* (esperamos 400K filas y 8 columnas)

In [22]:
print("Top 5 rows from s3://{}/{}/train/".format(bucket, output_prefix))
!aws s3 cp --quiet s3://{bucket}/{output_prefix}/train/train_features.csv - | head -n5

Top 5 rows from s3://sagemaker-us-east-1-561137843164/sagemaker/1c-processing/input/preprocessed/train/
12,0,30,40.0,0.0,0.0,0.0,0.0
13,0,30,40.0,0.0,0.0,0.0,20.0
14,0,30,40.0,0.0,0.0,0.0,0.0
15,0,30,40.0,0.0,0.0,0.0,0.0
16,0,30,40.0,0.0,0.0,0.0,0.0


In [21]:
import pandas as pd

s3_train_features = "s3://{}/{}/train/train_features.csv".format(bucket, output_prefix)
df_train = pd.read_csv(s3_train_features, header=None)
print("Train features shape:", df_train.shape)
df_train.head()

Train features shape: (8906058, 8)


,0,1,2,3,4,5,6,7
0,12,0,30,40.0,0.0,0.0,0.0,0.0
1,13,0,30,40.0,0.0,0.0,0.0,20.0
2,14,0,30,40.0,0.0,0.0,0.0,0.0
3,15,0,30,40.0,0.0,0.0,0.0,0.0
4,16,0,30,40.0,0.0,0.0,0.0,0.0


In [23]:
s3_test_features = "s3://{}/{}/test/test_features.csv".format(bucket, output_prefix)
df_test = pd.read_csv(s3_test_features, header=None)
print("Test features shape:", df_test.shape)
df_test.head()

Test features shape: (424098, 8)


,0,1,2,3,4,5,6,7
0,33,0,30,40.0,0.0,0.0,0.0,0.0
1,33,0,31,40.0,0.0,0.0,0.0,0.0
2,33,0,32,40.0,0.0,0.0,0.0,0.0
3,33,0,33,40.0,0.0,0.0,0.0,0.0
4,33,0,35,40.0,0.0,0.0,0.0,0.0


Los archivos de salida pueden usarse directamente como input para un training job.